In [87]:
import pandas as pd
# Loading Urdu data (Assuming it's loaded into `urdu_data` DataFrame)
urdu_data = pd.read_pickle('data.pkl')

In [88]:
urdu_data.head()

,Text,Type,Short Summary
0,دُنیا کی تاریخ زمین اور سمندروں پر ہونے والے ت...,History,سمندر کی گہرائیوں میں پائی جانے والی یہ معدنیا...
1,عام طور پر 5-6 ملی میٹر کی پتھری پیشاب کے ذریع...,Health,تاہم اگر مثانے میں پتھری ہو یا سائز میں چھوٹی ...
2,کوڑے کیسے شروع ہوئے؟\nشروع میں دالوں کو پیس کر...,History,کئی جگہوں پر اس کا ذکر بھی آیا ہے کہ پکوڑے یا ...
3,پپیتا نرم و ملائم جلد رکھنے والا پھل ہے۔یہ ابت...,Nature,پپیتا نرم و ملائم جلد رکھنے والا پھل ہے۔
4,دل کو تقویت دیتا ہے۔قے اور بخار سے نجات دلاتا ...,General,دل کو تقویت دیتا ہے۔


# lstm

In [89]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import spacy
import pandas as pd
# Loading Urdu data (Assuming it's loaded into `urdu_data` DataFrame)
urdu_data = pd.read_pickle('data.pkl')

# Use spaCy for sentence tokenization
nlp = spacy.blank('ur')  # Ensure you have the correct language model
nlp.add_pipe('sentencizer')

def tokenize_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]

# Tokenize all documents into sentences
urdu_data['sentences'] = urdu_data['Text'].apply(tokenize_sentences)

# Flatten all sentences into a single list to build vocabulary and sequence data
all_sentences = [sent for sublist in urdu_data['sentences'] for sent in sublist]

# Initialize and fit the tokenizer
tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')
tokenizer.fit_on_texts(all_sentences)

# Convert sentences to sequences
sentence_sequences = [tokenizer.texts_to_sequences(sents) for sents in urdu_data['sentences']]

# Pad sequences
max_length = max(len(seq) for sublist in sentence_sequences for seq in sublist)
padded_sequences = [pad_sequences(sublist, maxlen=max_length, padding='post') for sublist in sentence_sequences]

# Create a flat list of sequences and a corresponding label list
X = np.concatenate(padded_sequences)
labels = []

# Create labels based on inclusion in the short summary
for idx, row in urdu_data.iterrows():
    summary_sentences = tokenize_sentences(row['Short Summary'])
    document_sentences = row['sentences']
    labels += [1 if any(sen in doc for sen in summary_sentences) else 0 for doc in document_sentences]

y = np.array(labels)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [90]:
max_length

192

In [91]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Bidirectional, Dropout

# Model configuration
embedding_dim = 100

# Building the model
model = Sequential([
    Embedding(input_dim=5000, output_dim=embedding_dim, input_length=max_length),
    Bidirectional(LSTM(64, return_sequences=False)),  # Return sequences to get output for each input
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 192, 100)          500000    
                                                                 
 bidirectional (Bidirectiona  (None, 128)              84480     
 l)                                                              
                                                                 
 dropout_260 (Dropout)       (None, 128)               0         
                                                                 
 dense_8 (Dense)             (None, 1)                 129       
                                                                 
Total params: 584,609
Trainable params: 584,609
Non-trainable params: 0
_________________________________________________________________


In [139]:
# Training the model
model.fit(X_train, y_train, epochs=3, validation_split=0.2)

Epoch 1/3
95/95 [==============================] - 9s 80ms/step - loss: 0.0122 - accuracy: 0.9937 - val_loss: 0.0207 - val_accuracy: 0.9921
Epoch 2/3
95/95 [==============================] - 8s 83ms/step - loss: 0.0121 - accuracy: 0.9950 - val_loss: 0.0251 - val_accuracy: 0.9854
Epoch 3/3
95/95 [==============================] - 8s 86ms/step - loss: 0.0130 - accuracy: 0.9947 - val_loss: 0.0197 - val_accuracy: 0.9868


In [127]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

24/24 [==============================] - 1s 23ms/step - loss: 0.0146 - accuracy: 0.9934
Test Loss: 0.014573264867067337
Test Accuracy: 0.9933775067329407


In [138]:
model.save('LSTM_Summarizer.h5')

model = tf.keras.models.load_model('LSTM_Summarizer.h5')

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)

# Make predictions
predictions = model.predict(X_test)

24/24 [==============================] - 1s 20ms/step


In [140]:
def generate_summary(text, model, tokenizer, max_length):
    # Tokenize text
    nlp = spacy.blank('ur')  # Load the appropriate language model
    nlp.add_pipe('sentencizer')
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]

    # Convert sentences to sequences and pad them
    sequences = tokenizer.texts_to_sequences(sentences)
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

    # Get predictions from the model
    predictions = model.predict(padded_sequences)
    predicted_sentences = [s for s, p in zip(sentences, predictions) if p > 0.5]

    # Combine the selected sentences to form a summary
    return ' '.join(predicted_sentences)

# Example usage
new_text = urdu_data['Text'][510]
summary = predict_summary(new_text, tokenizer, model, max_length)
print('Text',new_text)
print("Predicted Summary:", summary)
print('Orignal Summary',urdu_data['Short Summary'][510])


1/1 [==============================] - 0s 19ms/step
Text ایک عام تاثر یہ ہے کہ سمارٹ فونز کی وجہ سے لوگ اپنی سماجی سرگرمیوں سے دور ہوتے جا رہے ہیں۔ لیکن اگر آپ اس پہلو پر ذرا غور کریں تو آپ کو اندازہ ہوگا کہ سمارٹ فونز سماجی دنیا سے آپ کا رابطہ اور ہموار کر رہے ہیں نا کہ کم کر رہے ہیں۔اور اب تو یہ بات تحقیق سے بھی ثابت ہو چکی ہے ۔ آپ کو ایسا لگتا ہوگا کہ سمارٹ فونز لوگوں کو ان کے ارد گرد ہونے والی واقعات سے بے خبر کر دیتے ہیں مگر حقیقیت اس کے برعکس ہے۔

2008 میں کیتھ ہیپٹن نے اس بارے میں تحقیق کرنے کے لیے نیو یارک کے برینٹ پارک او ر آرٹ کے میٹروپولیٹن میوزیم میں کچھ کیمرے لگوائے اور ان جگہوں کا 1970کی فوٹیج سے موازنہ کیا۔ان جگہوں پر لوگوں کا مشاہدہ کرنے کے بعد وہ اس نتیجے پر پہنچی کہ سمارٹ فونز گروپس کی شکل میں استعمال کیے جاتے ہیں۔
Predicted Summary: ایک عام تاثر یہ ہے کہ سمارٹ فونز کی وجہ سے لوگ اپنی سماجی سرگرمیوں سے دور ہوتے جا رہے ہیں۔
Orignal Summary ایک عام تاثر یہ ہے کہ سمارٹ فونز کی وجہ سے لوگ اپنی سماجی سرگرمیوں سے دور ہوتے جا رہے ہیں۔


In [142]:
from sklearn.metrics import classification_report

# Predict on test data
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Generate classification report
report = classification_report(y_test, y_pred, target_names=['Not in Summary', 'In Summary'])
print(report)

24/24 [==============================] - 0s 19ms/step
                precision    recall  f1-score   support

Not in Summary       1.00      0.99      1.00       570
    In Summary       0.98      0.99      0.99       185

      accuracy                           0.99       755
     macro avg       0.99      0.99      0.99       755
  weighted avg       0.99      0.99      0.99       755



# Rouge_1= 2*Recall*Precision/(Recall+Precision)

In [143]:
from rouge_score import rouge_scorer

def evaluate(references, predictions):
    # Initialize the ROUGE scorer
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    # Dictionary to store the scores
    scores = {'rouge1': {'precision': 0, 'recall': 0, 'fmeasure': 0},
              'rouge2': {'precision': 0, 'recall': 0, 'fmeasure': 0},
              'rougeL': {'precision': 0, 'recall': 0, 'fmeasure': 0}}
    
    # Calculate scores for each pair of reference and prediction
    for reference, prediction in zip(references, predictions):
        score = scorer.score(reference, prediction)
        for key in scores:
            scores[key]['precision'] += score[key].precision
            scores[key]['recall'] += score[key].recall
            scores[key]['fmeasure'] += score[key].fmeasure
    
    # Average the scores
    num_scores = len(predictions)
    for key in scores:
        scores[key]['precision'] /= num_scores
        scores[key]['recall'] /= num_scores
        scores[key]['fmeasure'] /= num_scores
    
    return scores

# Example usage
references = urdu_data['Text'][950:1000]
predictions = urdu_data['Short Summary'][950:1000]
evaluations = evaluate(references, predictions)
evaluations

{'rouge1': {'precision': 0.18,
  'recall': 0.058337164750957855,
  'fmeasure': 0.07972650878533233},
 'rouge2': {'precision': 0.12,
  'recall': 0.017710084033613446,
  'fmeasure': 0.02988888888888889},
 'rougeL': {'precision': 0.18,
  'recall': 0.058337164750957855,
  'fmeasure': 0.07972650878533233}}

In [144]:
def calculate_rouge_scores(rouge_scores):
    rouge_1_precision = rouge_scores['rouge1']['precision']
    rouge_1_recall = rouge_scores['rouge1']['recall']
    rouge_1_f1 = (2 * rouge_1_precision * rouge_1_recall) / (rouge_1_precision + rouge_1_recall)
    
    rouge_2_precision = rouge_scores['rouge2']['precision']
    rouge_2_recall = rouge_scores['rouge2']['recall']
    rouge_2_f1 = (2 * rouge_2_precision * rouge_2_recall) / (rouge_2_precision + rouge_2_recall)
    
    rouge_l_precision = rouge_scores['rougeL']['precision']
    rouge_l_recall = rouge_scores['rougeL']['recall']
    rouge_l_f1 = (2 * rouge_l_precision * rouge_l_recall) / (rouge_l_precision + rouge_l_recall)
    
    return rouge_1_f1, rouge_2_f1, rouge_l_f1



rouge_1_f1, rouge_2_f1, rouge_l_f1 = calculate_rouge_scores(evaluations)

print("ROUGE-1 F1-score:", rouge_1_f1)
print("ROUGE-2 F1-score:", rouge_2_f1)
print("ROUGE-L F1-score:", rouge_l_f1)


ROUGE-1 F1-score: 0.08811625888177989
ROUGE-2 F1-score: 0.030864988558352402
ROUGE-L F1-score: 0.08811625888177989


# Customize BERT

In [27]:
import tensorflow as tf
from transformers import TFBertModel, TFBertTokenizer, AutoTokenizer
from tensorflow.keras.layers import Input, Dense, Dropout, Attention
from tensorflow.keras.models import Model
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from torch.utils.data import DataLoader, Dataset

In [35]:
# Load the dataset
file_path = './data.pkl'
data = pd.read_pickle(file_path)
data.head()

,Text,Type,Short Summary
0,دُنیا کی تاریخ زمین اور سمندروں پر ہونے والے ت...,History,سمندر کی گہرائیوں میں پائی جانے والی یہ معدنیا...
1,عام طور پر 5-6 ملی میٹر کی پتھری پیشاب کے ذریع...,Health,تاہم اگر مثانے میں پتھری ہو یا سائز میں چھوٹی ...
2,کوڑے کیسے شروع ہوئے؟\nشروع میں دالوں کو پیس کر...,History,کئی جگہوں پر اس کا ذکر بھی آیا ہے کہ پکوڑے یا ...
3,پپیتا نرم و ملائم جلد رکھنے والا پھل ہے۔یہ ابت...,Nature,پپیتا نرم و ملائم جلد رکھنے والا پھل ہے۔
4,دل کو تقویت دیتا ہے۔قے اور بخار سے نجات دلاتا ...,General,دل کو تقویت دیتا ہے۔


In [33]:
from transformers import AutoTokenizer, TFBertModel
import tensorflow as tf

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = TFBertModel.from_pretrained("google-bert/bert-base-uncased")

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [36]:
# Function to tokenize the data
def tokenize_and_prepare_masks(data, max_length):
    # Tokenizing the data
    tokens = tokenizer(
        data['Text'].tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors="np"
    )
    
    return tokens['input_ids'], tokens['attention_mask']

# Maximum sequence length
max_length = 512

# Tokenize data and prepare attention masks
input_ids, attention_masks = tokenize_and_prepare_masks(data, max_length)
input_ids.shape, attention_masks.shape

((1000, 512), (1000, 512))

In [45]:
# Assuming `data` is your DataFrame
# Example function to generate labels for each text entry matching its tokens
def generate_labels(text, summary, tokenizer, max_length):
    tokenized_text = tokenizer.encode(text, add_special_tokens=False)
    tokenized_summary = tokenizer.encode(summary, add_special_tokens=False)
    
    # Initialize labels with 0 (not in summary)
    labels = [0] * len(tokenized_text)
    
    # Set labels to 1 for tokens that appear in the summary
    summary_indices = [i for i, token in enumerate(tokenized_text) if token in tokenized_summary]
    for idx in summary_indices:
        labels[idx] = 1
    
    # Ensure labels match the required max_length
    if len(labels) > max_length:
        labels = labels[:max_length]
    else:
        labels += [0] * (max_length - len(labels))
    
    return labels

# Applying label creation across the dataset
data['labels'] = data.apply(lambda x: generate_labels(x['Text'], x['Short Summary'], tokenizer, max_length), axis=1)

# Convert labels list to appropriate format for training
labels = np.array(data['labels'].tolist())


Token indices sequence length is longer than the specified maximum sequence length for this model (615 > 512). Running this sequence through the model will result in indexing errors


In [46]:
labels

array([[1, 1, 1, ..., 1, 1, 1],
       [0, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 1, 1, 1],
       ...,
       [1, 1, 0, ..., 1, 1, 1],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]])

In [47]:
from sklearn.model_selection import train_test_split

# Split input_ids and labels
x_train_ids, x_val_ids, y_train, y_val = train_test_split(
    input_ids, labels, test_size=0.1, random_state=42
)

# Split attention_masks using the same indices
_, x_val_masks = train_test_split(
    attention_masks, test_size=0.1, random_state=42
)

# Now x_val_ids and x_val_masks are aligned with y_val

In [92]:
from transformers import TFBertModel
from keras.layers import Input, Dense, Dropout, Attention
from keras.models import Model

def create_extractive_summarization_model(max_length):
    # Define input layers
    input_ids = Input(shape=(max_length,), dtype=tf.int32)
    attention_mask = Input(shape=(max_length,), dtype=tf.int32)
    
    # Load pre-trained BERT model
    bert_model = TFBertModel.from_pretrained('bert-base-uncased')
    
    # BERT-based encoder
    outputs = bert_model(input_ids, attention_mask=attention_mask)[0]
    
    # Attention mechanism
    attention_output = Attention()([outputs, outputs])
    
    # Dense layer
    dense_layer = Dense(units=max_length, activation='relu')(attention_output)
    
    # Output layer
    output = Dense(units=1, activation='sigmoid')(dense_layer)
    
    # Define model
    model = Model(inputs=[input_ids, attention_mask], outputs=output)
    
    return model

# Maximum sequence length
max_length = 192

# Create the model
model = create_extractive_summarization_model(max_length)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_13 (InputLayer)          [(None, 192)]        0           []                               
                                                                                                  
 input_14 (InputLayer)          [(None, 192)]        0           []                               
                                                                                                  
 tf_bert_model_7 (TFBertModel)  TFBaseModelOutputWi  109482240   ['input_13[0][0]',               
                                thPoolingAndCrossAt               'input_14[0][0]']               
                                tentions(last_hidde                                               
                                n_state=(None, 192,                                         

In [49]:

# Train the model
history = model.fit(
    [x_train_ids, _], y_train,
    validation_data=([x_val_ids, x_val_masks], y_val),
    epochs=3, batch_size=batch_size, callbacks=[tensorboard_callback]
)


Epoch 1/3
57/57 [==============================] - 8909s 156s/step - loss: 0.8226 - accuracy: 0.5953 - val_loss: 0.6643 - val_accuracy: 0.6292
Epoch 2/3
57/57 [==============================] - 8608s 151s/step - loss: 0.6680 - accuracy: 0.6268 - val_loss: 0.6667 - val_accuracy: 0.6292
Epoch 3/3
57/57 [==============================] - 8951s 158s/step - loss: 0.6631 - accuracy: 0.6385 - val_loss: 0.6908 - val_accuracy: 0.6292


In [53]:
# Save the model
model_path = "./BERT MODIFIED/"
model.save(model_path+'model.h5')


In [57]:
from rouge_score import rouge_scorer

def calculate_rouge_scores(references, hypotheses):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = {
        'rouge1': [],
        'rouge2': [],
        'rougeL': []
    }

    for ref, hyp in zip(references, hypotheses):
        score = scorer.score(ref, hyp)
        scores['rouge1'].append(score['rouge1'].fmeasure)
        scores['rouge2'].append(score['rouge2'].fmeasure)
        scores['rougeL'].append(score['rougeL'].fmeasure)

    # Calculate average scores
    avg_scores = {key: np.mean(val) for key, val in scores.items()}
    return avg_scores


In [61]:
def tokenize_and_prepare_masks(texts, max_length):
    tokens = tokenizer(
        texts,  # Directly use the list of texts
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors="np"
    )
    return tokens['input_ids'], tokens['attention_mask']

# Assuming test_text is already a list of strings as extracted from the DataFrame
test_input_ids, test_attention_masks = tokenize_and_prepare_masks(test_text, max_length)


In [63]:
def generate_summary(input_ids, attention_mask, model, tokenizer):
    predictions = model.predict([input_ids, attention_mask])
    summaries = []
    for i, prediction in enumerate(predictions):
        # Thresholding the predictions to classify as 0 or 1 (not included/included)
        thresholded_prediction = (prediction > 0.5).astype(int)
        # Decoding tokens to text, only including tokens where prediction == 1
        summary_tokens = [tok for j, tok in enumerate(input_ids[i]) if thresholded_prediction[j] == 1]
        summary_text = tokenizer.decode(summary_tokens)
        summaries.append(summary_text)
    return summaries

In [64]:
predicted_summaries = generate_summary(test_input_ids, test_attention_masks, model, tokenizer)

2/2 [==============================] - 29s 9s/step


In [67]:
# Calculate ROUGE scores
rouge_scores = calculate_rouge_scores(test_summaries, predicted_summaries)
print("ROUGE Scores:", rouge_scores)

ROUGE Scores: {'rouge1': 0.024028352029151708, 'rouge2': 0.01023507565880447, 'rougeL': 0.024028352029151708}


## For Testing

In [68]:
def preprocess_and_tokenize(text, tokenizer, max_length=512):
    # Tokenizing the single text input
    tokens = tokenizer(
        text,  # Direct text input
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors="tf"  # Ensure this is set to TensorFlow tensors for the model compatibility
    )
    return tokens['input_ids'], tokens['attention_mask']


In [73]:
def generate_summary(input_ids, attention_mask, model, tokenizer):
    # Get model predictions
    predictions = model.predict([input_ids, attention_mask])[0]  # Assuming the model outputs logits

    # Apply threshold to get binary mask
    thresholded_prediction = (predictions > 0.5).astype(int)

    # Convert to flat numpy array if necessary
    thresholded_prediction = thresholded_prediction.flatten()

    # Extract the tokens that are part of the summary
    summary_tokens = [tok for idx, tok in enumerate(input_ids[0]) if thresholded_prediction[idx] == 1]

    # Decode the selected tokens to text
    summary_text = tokenizer.decode(summary_tokens)
    return summary_text


In [70]:
def create_summary_from_text(text, tokenizer, model, max_length=512):
    # Preprocess and tokenize the input text
    input_ids, attention_mask = preprocess_and_tokenize(text, tokenizer, max_length)
    
    # Generate the summary
    summary = generate_summary(input_ids, attention_mask, model, tokenizer)
    
    return summary


In [76]:
i = 50
text = data['Text'][i]
orig_sum = data['Short Summary'][i]


user_input_text = text
# Assuming `model` and `tokenizer` are already loaded and available
summary_text = create_summary_from_text(user_input_text, tokenizer, model)
print("Actuall Text", text)
print("Generated Summary:", summary_text)
print("Orignal Summary", orig_sum)

1/1 [==============================] - 1s 862ms/step
Actuall Text کراچی (اُردو پوائنٹ اخبارتازہ ترین - این این آئی۔ 17 اپریل2024ء) معروف اداکارہ نازش جہانگیر کا کہنا ہے کہ درفشاں سلیم ٹیلنٹ کے بجائے قسمت سے مقبول ہو رہی ہیں اور وہ جس ڈرامے میں آتی ہیں وہ کامیاب ہوجاتا ہے۔ایک ٹی وی شو میں ایک سوال کے جواب میں اداکارہ کا کہنا تھا کہ آغا علی اور خوشحال خان کو بولی وڈ میں ہونا چاہیے۔اداکارہ نے بتایا تھا کہ اگر انہیں موقع ملے تو وہ عائزہ خان سے ان کا گھر لے لیں گی جب کہ سارہ خان سے وہ شوہر کی جانب سے تحفے میں ملنے والی گاڑیاں لے لیں گی۔


نازش جہانگیر کا کہنا تھا کہ وہ ماہرہ خان سے ان کی شخصیت لیں گی جب کہ ان کی نظر میں اقرا عزیز ایک اچھی بیوی ہیں۔اداکارہ میرا کے حوالے سے پوچھے گئے سوال پر نازش جہانگیر کا کہنا تھا کہ وہ ان سے کچھ نہیں لیں گی البتہ انہیں ہر وہ چیز دیں گی جو اداکارہ ان سے مانگیں گی۔اداکارہ سے جب پوچھا گیا کہ وہ کون سی اداکارہ ہیں جو ٹیلنٹ کے بجائے قسمت سے زیادہ مقبول ہوئیں اس پر نازش جہانگیر نے اداکارہ درفشاں سلیم کا نام لیا اور مزید کہا کہ وہ ٹیلنٹ سے زیادہ قسمت سے کامیاب 

In [77]:
i = 50
text = """یاد رکھیں، اس فنکشن کی مجموعی درستگی اور کارکردگی کا بہت زیادہ انحصار اس بات پر ہوگا کہ ماڈل کو ٹوکن کی سطح پر بائنری درجہ بندی کرنے کے لیے کتنی اچھی تربیت دی گئی ہے (یعنی یہ فیصلہ کرنا کہ کون سے ٹوکنز کو سمری میں شامل کیا جانا چاہیے)۔ بہتر نتائج کے لیے، اس بات کو یقینی بنائیں کہ ماڈل ٹریننگ اس خلاصہ کے کام کے ساتھ اچھی طرح سے ہم آہنگ ہو۔"""

user_input_text = text
# Assuming `model` and `tokenizer` are already loaded and available
summary_text = create_summary_from_text(user_input_text, tokenizer, model)
print("Actuall Text", text)
print("Generated Summary:", summary_text)

1/1 [==============================] - 1s 854ms/step
Actuall Text یاد رکھیں، اس فنکشن کی مجموعی درستگی اور کارکردگی کا بہت زیادہ انحصار اس بات پر ہوگا کہ ماڈل کو ٹوکن کی سطح پر بائنری درجہ بندی کرنے کے لیے کتنی اچھی تربیت دی گئی ہے (یعنی یہ فیصلہ کرنا کہ کون سے ٹوکنز کو سمری میں شامل کیا جانا چاہیے)۔ بہتر نتائج کے لیے، اس بات کو یقینی بنائیں کہ ماڈل ٹریننگ اس خلاصہ کے کام کے ساتھ اچھی طرح سے ہم آہنگ ہو۔
Generated Summary: [CLS] یاد رکھیں ، اس فنکشن کی مجموعی درستگی اور کارکردگی کا بہت زیادہ انحصار اس بات پر ہوگا کہ [UNK] کو ٹوکن کی سطح پر باينری درجہ بندی کرنے کے لیے کتنی اچھی تربیت دی گيی ہے ( یعنی یہ فیصلہ کرنا کہ کون سے ٹوکنز کو سمری میں شامل کیا جانا چاہیے ) [UNK] بہتر نتايج کے لیے ، اس بات کو یقینی بنايیں کہ [UNK] ٹریننگ اس خلاصہ کے کام کے ساتھ اچھی طرح سے ہم اہنگ ہو [UNK] [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [